Calculate the error propagataion for each typology class sing Monte Carlo Simulation

In [ ]:
import pandas as pd
import numpy as np
import camelot  #extract tabular data from PDFs into data formats like csv, excel, json, or pandas dataframes
from collections import Counter

from pathlib import Path
from collections import Counter   #count occurences of items in a list, string, or other iterable.  #frequency analysis, text processing, stats

Error matrices of CDL for Great Lakes

In [ ]:
base = Path(r"great_lakes_typology/cdl_accuracy_assessments")

states = ["MN", "WI", "IL", "MI", "IN", "OH", "PA", "NY"]
years = range(2008, 2024)

for year in years:

    yy = str(year)[-2:]
    folder = base / f"CDL_{year}_accuracy_assessments"

    summaries = []
    matrices = []

    for state in states:

        file = list(folder.glob(f"NASS_CDL_{state}{yy}_accuracy*"))[0]  #*means anything can follow that text  [0] takes the first matching file

        # Summary Table
        df = pd.read_excel(file, sheet_name="Summary Table", header=10)

        df = df.iloc[:, [0, 1, 4, 5, 8, 9, 11]]
        df.columns = [
            "Category", "Code",
            "Producer", "Omission",
            "User", "Commission", "Bias"
        ]

        df["Category"] = df["Category"].astype(str).str.strip()
        df["Code"] = pd.to_numeric(df["Code"], errors="coerce")

        df = df.dropna(subset=["Code"])

        summaries.append(df)

        # All - Matrix
        m = pd.read_excel(file, sheet_name="All - Matrix", header=None)

        codes = pd.to_numeric(m.iloc[0, 2:258], errors="coerce")

        matrix = (
            m.iloc[1:257, 2:258]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0)
        )

        matrix.index = codes
        matrix.columns = codes

        # Convert each row to probabilities
        matrix = matrix.div(
            matrix.sum(axis=1).replace(0, np.nan),
            axis=0
        )

        matrices.append(matrix)

    # Average Summary Table
    avg_summary = (
        pd.concat(summaries)
        .groupby(["Code", "Category"], as_index=False)
        [["Producer", "Omission", "User", "Commission", "Bias"]]
        .mean()
    )

    # Average Error Matrix
    avg_matrix = (
        pd.concat(matrices)
        .groupby(level=0)
        .mean()
    )

    # Save
    output = f"great_lakes_typology/cdl_accuracy_assessments/great_lakes_acuracy_assessments/great_lakes_cdl_accuracy_{year}.xlsx"

    with pd.ExcelWriter(output) as writer:

        avg_summary.to_excel(
            writer,
            sheet_name="Average Summary",
            index=False
        )

        avg_matrix.to_excel(
            writer,
            sheet_name="Average Error Matrix"
        )

    print(f"Finished {year}")

Rename the class codes from the CDL matrix file to class names

In [ ]:
code_file = r"great_lakes_typology/monte_carlo_simulation/samples/crop_codes_new.xlsx"

cdl_folder = "great_lakes_typology/monte_carlo_simulation/cdl_accuracy_assessments/great_lakes_cdl_acuracy_assessments"

# Code -> Crop Name
codes = pd.read_excel(code_file, sheet_name="CDL")

codes["Code"] = pd.to_numeric(codes["Code"], errors="coerce")
codes = codes.dropna(subset=["Code", "Crop_Name"])

code_to_name = dict(
    zip(
        codes["Code"].astype(int),
        codes["Crop_Name"].astype(str).str.strip()
    )
)

for year in range(2008, 2024):

    file = f"{cdl_folder}/great_lakes_cdl_accuracy_{year}.xlsx"

    matrix = pd.read_excel(
        file,
        sheet_name="Average Error Matrix",
        index_col=0
    )

    # Remove completely empty rows and columns
    matrix = matrix.dropna(axis=0, how="all")
    matrix = matrix.dropna(axis=1, how="all")

    # Rename crop codes to crop names
    matrix.index = [
        code_to_name.get(int(float(x)), x)
        for x in matrix.index
    ]

    matrix.columns = [
        code_to_name.get(int(float(x)), x)
        for x in matrix.columns
    ]

    # Replace existing sheet
    with pd.ExcelWriter(
        file,
        mode="a",
        engine="openpyxl",
        if_sheet_exists="replace"
    ) as writer:

        matrix.to_excel(
            writer,
            sheet_name="Average Error Matrix"
        )

    print(f"Finished {year}")

Error Matrices of NLCD for Great Lakes

In [ ]:
pdf = r"great_lakes_typology/nlcd_accuracy_assessments/annual_nlcd_cu_c1v0_vs_lcnext_strs_8360_validation_report_508.pdf"
out = Path(r"great_lakes_typology/nlcd_accuracy_assessments/conus_nlcd_accuracy")

classes = {
    11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands"
}

out.mkdir(exist_ok=True)

# PDF pages 30–45 = Level 2 tables for 2008–2023
tables = camelot.read_pdf(pdf, pages="30-45", flavor="stream")

for table in tables:

    df = table.df
    year = int(df.iloc[0, 0].split(",")[1])

    codes = df.iloc[3:19, 0].astype(int).tolist()
    names = [classes[c] for c in codes]

    # Confusion matrix
    matrix = df.iloc[3:19, 1:17].replace("", 0).astype(float)
    matrix.index = names
    matrix.columns = names

    # Convert each mapped class row to Monte Carlo probabilities
    mc_matrix = matrix.div(matrix.sum(axis=1), axis=0)

    # User's accuracy
    ua = df.iloc[3:19, 19].str.extract(r"([\d.]+)")[0].astype(float).values

    # Producer's accuracy
    pa = df.iloc[21, 1:17].str.extract(r"([\d.]+)")[0].astype(float).values

    bias = pd.to_numeric(df.iloc[3:19, 18]).values

    summary = pd.DataFrame({
        "Code": codes,
        "Class": names,
        "User_Accuracy": ua,
        "Producer_Accuracy": pa,
        "Bias": bias
    })

    with pd.ExcelWriter(out / f"nlcd_accuracy_{year}.xlsx") as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        mc_matrix.to_excel(writer, sheet_name="MC Matrix")

    print(f"Finished {year}")

Perform the error propagation using monte carlo simulation

In [ ]:
# Rule Functions for Crop-Sequence Pattern Detection

def is_monoculture(sequence, target_crop, threshold=0.85):
    clean_seq = [g for g in sequence if g and g != 'nan']
    freq = clean_seq.count(target_crop) / len(clean_seq) if len(clean_seq) > 0 else 0
    return freq >= threshold

def is_staple_crop_rotation(sequence):
    staple_crops = {'Corn', 'Wheat', 'Soybeans'}
    clean_seq = [g for g in sequence if g not in {'nan', '', None}]
    staple_count = sum(1 for g in clean_seq if g in staple_crops)
    return (staple_count / len(clean_seq)) >= 0.75 and len(set(clean_seq)) > 1

def is_staple_crop_conversion(sequence):
    staple_crops = {'Corn', 'Wheat', 'Soybeans'}
    non_staple_cropland = {
        'Grains', 'Oilseeds', 'Legumes and Pulses', 'Forage and Cover Crops', 
        'Vegetables', 'Fruits and Nuts', 'Other Crops', 'Specialty Crops',
        'Sugar and Fiber Crops', 'Double Crops', 'Fallowing', 
        'Urban Area', 'Full Solar', 'Partial Solar', 'Forest', 
        'Pasture/Hay', 'Shrubland','Waterbody', 'Barren Land', 'Other Land'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_staple_ratio = sum(1 for g in start_seq if g in staple_crops) / len(start_seq)
    start_non_staple_ratio = sum(1 for g in start_seq if g in non_staple_cropland) / len(start_seq)
    end_staple_ratio = sum(1 for g in end_seq if g in staple_crops) / len(end_seq)

    return start_staple_ratio <= 0.3636 and start_non_staple_ratio >= 0.6364 and end_staple_ratio >= 0.6

def is_other_cropland_rotation(sequence):
    staple_crops = {'Corn', 'Wheat', 'Soybeans'}
    other_crops = {
        'Grains', 'Oilseeds', 'Legumes and Pulses', 'Forage and Cover Crops',
        'Vegetables', 'Fruits and Nuts', 'Other Crops', 'Specialty Crops',
        'Sugar and Fiber Crops', 'Double Crops', 'Fallowing'
    }
    
    clean_seq = [g for g in sequence if g not in {'nan', '', None}]
    if not clean_seq:
        return False

    other_ratio = sum(1 for g in clean_seq if g in other_crops) / len(clean_seq)
    has_staple = any(g in staple_crops for g in clean_seq)
    has_other_land = any(g not in (staple_crops | other_crops) for g in clean_seq)

    return other_ratio >= 0.75 and (has_staple or has_other_land)

def is_non_staple_crop_conversion(sequence):
    staple_and_other_land = {
        'Corn', 'Wheat', 'Soybeans', 'Urban Area', 'Full Solar', 
        'Partial Solar', 'Forest', 'Pasture/Hay', 'Shrubland', 
        'Waterbody', 'Barren Land', 'Other Land'
    }
    non_staple_cropland = {
        'Grains', 'Oilseeds', 'Legumes and Pulses', 'Forage and Cover Crops', 
        'Vegetables', 'Fruits and Nuts', 'Other Crops', 'Specialty Crops',
        'Sugar and Fiber Crops', 'Double Crops', 'Fallowing'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_staple_other_ratio = sum(1 for g in start_seq if g in staple_and_other_land) / len(start_seq)
    end_staple_other_ratio = sum(1 for g in end_seq if g in staple_and_other_land) / len(end_seq)
    start_non_staple_ratio = sum(1 for g in start_seq if g in non_staple_cropland) / len(start_seq)

    return start_staple_other_ratio <= 0.3636 and start_non_staple_ratio >= 0.6364 and end_staple_other_ratio >= 0.6

def is_cropland_intensification(sequence):
    non_cropland_set = {
        'Urban Area', 'Full Solar', 'Partial Solar', 'Forest', 
        'Pasture/Hay', 'Shrubland', 'Waterbody', 'Barren Land', 'Other Land'
    }
    cropland_set = {
        'Grains', 'Oilseeds', 'Legumes and Pulses', 'Forage and Cover Crops', 
        'Vegetables', 'Fruits and Nuts', 'Other Crops', 'Specialty Crops',
        'Sugar and Fiber Crops', 'Double Crops', 'Fallowing',
        'Corn', 'Soybeans', 'Wheat'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_non_cropland_ratio = sum(1 for g in start_seq if g in non_cropland_set) / len(start_seq)
    start_cropland_ratio = sum(1 for g in start_seq if g in cropland_set) / len(start_seq)
    end_cropland_ratio = sum(1 for g in end_seq if g in cropland_set) / len(end_seq)

    return start_non_cropland_ratio >= 0.6364 and start_cropland_ratio <= 0.3636 and end_cropland_ratio >= 0.6

def is_cropland_reduction(sequence):
    cropland_set = {
        'Corn', 'Wheat', 'Soybeans', 'Grains', 'Oilseeds', 'Legumes and Pulses',
        'Forage and Cover Crops', 'Vegetables', 'Fruits and Nuts', 'Other Crops', 
        'Specialty Crops', 'Sugar and Fiber Crops', 'Double Crops', 'Fallowing'
    }
    non_cropland_set = {
        'Urban Area', 'Full Solar', 'Partial Solar', 'Forest', 'Pasture/Hay', 
        'Shrubland', 'Waterbody', 'Barren Land', 'Other Land'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_cropland_ratio = sum(1 for g in start_seq if g in cropland_set) / len(start_seq)
    start_non_cropland_ratio = sum(1 for g in start_seq if g in non_cropland_set) / len(start_seq)
    end_non_cropland_ratio = sum(1 for g in end_seq if g in non_cropland_set) / len(end_seq)

    return start_cropland_ratio >= 0.6364 and start_non_cropland_ratio <= 0.3636 and end_non_cropland_ratio >= 0.6

def is_pasture_conversion(sequence):
    pasture_set = {'Pasture/Hay'}
    non_pasture_set = {
        'Urban Area', 'Full Solar', 'Partial Solar', 'Forest', 'Shrubland',
        'Waterbody', 'Barren Land', 'Other Land', 'Grains', 'Oilseeds',
        'Legumes and Pulses', 'Forage and Cover Crops', 'Vegetables', 'Fruits and Nuts',
        'Other Crops', 'Specialty Crops', 'Sugar and Fiber Crops', 
        'Double Crops', 'Fallowing', 'Corn', 'Soybeans', 'Wheat'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_non_pasture_ratio = sum(1 for g in start_seq if g in non_pasture_set) / len(start_seq)
    start_pasture_ratio = sum(1 for g in start_seq if g in pasture_set) / len(start_seq)
    end_pasture_ratio = sum(1 for g in end_seq if g in pasture_set) / len(end_seq)

    return start_non_pasture_ratio >= 0.6364 and start_pasture_ratio <= 0.3636 and end_pasture_ratio >= 0.6

def is_pasture_persistence(sequence):
    pasture_set = {'Pasture/Hay'}
    return (sum(1 for g in sequence if g in pasture_set) / len(sequence)) >= 0.85

def is_urban_persistence(sequence):
    return (sequence.count('Urban Area') / len(sequence)) >= 0.85

def is_urban_conversion(sequence):
    urban_set = {'Urban Area'}
    non_urban_set = {
        'Full Solar', 'Partial Solar', 'Forest', 'Shrubland', 'Waterbody',
        'Barren Land', 'Other Land', 'Grains', 'Oilseeds', 'Legumes and Pulses',
        'Forage and Cover Crops', 'Vegetables', 'Fruits and Nuts', 'Other Crops',
        'Specialty Crops', 'Sugar and Fiber Crops', 'Double Crops', 'Fallowing',
        'Corn', 'Soybeans', 'Wheat', 'Pasture/Hay'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_non_urban_ratio = sum(1 for g in start_seq if g in non_urban_set) / len(start_seq)
    start_urban_ratio = sum(1 for g in start_seq if g in urban_set) / len(start_seq)
    end_urban_ratio = sum(1 for g in end_seq if g in urban_set) / len(end_seq)

    return start_non_urban_ratio >= 0.6364 and start_urban_ratio <= 0.3636 and end_urban_ratio >= 0.6

def is_natural_land_persistence(sequence):
    natural_set = {'Forest', 'Shrubland', 'Waterbody', 'Barren Land'}
    return (sum(1 for g in sequence if g in natural_set) / len(sequence)) >= 0.85

# Label Assignment Function

def assign_subtypology_label(sequence):
    sequence = [str(g).strip() for g in sequence if pd.notna(g) and str(g).strip() != '']
    if not sequence:
        return 'Other Mixed Pattern'

    if is_monoculture(sequence, 'Corn'):
        return 'Corn Monoculture'
    elif is_monoculture(sequence, 'Wheat'):
        return 'Wheat Monoculture'
    elif is_monoculture(sequence, 'Soybeans'):
        return 'Soybean Monoculture'
    elif is_staple_crop_rotation(sequence):
        return 'Staple Crop Rotation'
    elif is_other_cropland_rotation(sequence):
        return 'Other Cropland Rotation'
    elif is_urban_persistence(sequence):
        return 'Urban Persistence'
    elif is_cropland_intensification(sequence):
        return 'Cropland Intensification'
    elif is_cropland_reduction(sequence):
        return 'Cropland Reduction'
    elif is_natural_land_persistence(sequence):
        return 'Natural Land Persistence'
    elif is_staple_crop_conversion(sequence):
        return 'Staple Crop Conversion'
    elif is_non_staple_crop_conversion(sequence):
        return 'Other Cropland Conversion'
    elif is_pasture_conversion(sequence):
        return 'Pasture/Hay Conversion'
    elif is_pasture_persistence(sequence):
        return 'Pasture/Hay Persistence'
    elif is_urban_conversion(sequence):
        return 'Urban Area Conversion'
    else:
        return 'Other Mixed Pattern'

In [ ]:
samples_file = "great_lakes_typology/monte_carlo_simulation/raw_lulc_transition/lulc_transition_nlcd_cdl_samples.csv"

cdl_folder = "great_lakes_typology/monte_carlo_simulation/cdl_accuracy_assessments/great_lakes_cdl_acuracy_assessments"

nlcd_folder = "great_lakes_typology/monte_carlo_simulation/nlcd_accuracy_assessments/conus_nlcd_accuracy_assessments"

years = range(2008, 2024)
N_SIM = 1000

In [3]:
# Dictionary for CDL

cdl_group = {
    "Durum Wheat": "Wheat",
    "Winter Wheat": "Wheat",
    "Spring Wheat": "Wheat",
    "Corn": "Corn",
    "Soybeans": "Soybeans",

    "Barley": "Grains",
    "Sorghum": "Grains",
    "Millet": "Grains",
    "Speltz": "Grains",
    "Other Small Grains": "Grains",
    "Rice": "Grains",

    "Sunflower": "Oilseeds",
    "Canola": "Oilseeds",
    "Flaxseed": "Oilseeds",
    "Safflower": "Oilseeds",
    "Rape Seed": "Oilseeds",

    "Peanuts": "Legumes and Pulses",
    "Chick Peas": "Legumes and Pulses",
    "Lentils": "Legumes and Pulses",
    "Peas": "Legumes and Pulses",
    "Dry Beans": "Legumes and Pulses",

    "Camelina": "Forage and Cover Crops",
    "Mustard": "Forage and Cover Crops",
    "Buckwheat": "Forage and Cover Crops",
    "Triticale": "Forage and Cover Crops",
    "Vetch": "Forage and Cover Crops",
    "Oats": "Forage and Cover Crops",
    "Rye": "Forage and Cover Crops",
    "Alfalfa": "Forage and Cover Crops",
    "Other Hay/Non Alfalfa": "Forage and Cover Crops",
    "Clover/Wildflowers": "Forage and Cover Crops",
    "Sod/Grass Seed": "Forage and Cover Crops",
    "Switchgrass": "Forage and Cover Crops",

    "Sweet Corn": "Vegetables",
    "Tomatoes": "Vegetables",
    "Cucumbers": "Vegetables",
    "Carrots": "Vegetables",
    "Lettuce": "Vegetables",
    "Pumpkins": "Vegetables",
    "Broccoli": "Vegetables",
    "Cauliflower": "Vegetables",
    "Celery": "Vegetables",
    "Radishes": "Vegetables",
    "Turnips": "Vegetables",
    "Eggplants": "Vegetables",
    "Gourds": "Vegetables",
    "Cabbage": "Vegetables",
    "Squash": "Vegetables",
    "Greens": "Vegetables",
    "Asparagus": "Vegetables",
    "Garlic": "Vegetables",
    "Onions": "Vegetables",
    "Peppers": "Vegetables",
    "Potatoes": "Vegetables",
    "Sweet Potatoes": "Vegetables",
    "Sugarbeets": "Vegetables",

    "Cherries": "Fruits and Nuts",
    "Peaches": "Fruits and Nuts",
    "Apples": "Fruits and Nuts",
    "Grapes": "Fruits and Nuts",
    "Other Tree Crops": "Fruits and Nuts",
    "Citrus": "Fruits and Nuts",
    "Pecans": "Fruits and Nuts",
    "Almonds": "Fruits and Nuts",
    "Walnuts": "Fruits and Nuts",
    "Pears": "Fruits and Nuts",
    "Pistachios": "Fruits and Nuts",
    "Olives": "Fruits and Nuts",
    "Pomegranates": "Fruits and Nuts",
    "Nectarines": "Fruits and Nuts",
    "Plums": "Fruits and Nuts",
    "Apricots": "Fruits and Nuts",
    "Avocados": "Fruits and Nuts",
    "Blueberries": "Fruits and Nuts",
    "Cranberries": "Fruits and Nuts",
    "Strawberries": "Fruits and Nuts",
    "Prunes": "Fruits and Nuts",
    "Cantaloupes": "Fruits and Nuts",
    "Honeydew Melons": "Fruits and Nuts",
    "Oranges": "Fruits and Nuts",
    "Watermelons": "Fruits and Nuts",

    "Other Crops": "Other Crops",
    "Misc Vegs & Fruits": "Other Crops",

    "Mint": "Specialty Crops",
    "Hops": "Specialty Crops",
    "Herbs": "Specialty Crops",
    "Caneberries": "Specialty Crops",
    "Pop or Orn Corn": "Specialty Crops",

    "Sugarcane": "Sugar and Fiber Crops",
    "Cotton": "Sugar and Fiber Crops",
    "Tobacco": "Sugar and Fiber Crops",

    "Dbl Crop WinWht/Soybeans": "Double Crops",
    "Dbl Crop WinWht/Corn": "Double Crops",
    "Dbl Crop Oats/Corn": "Double Crops",
    "Dbl Crop Triticale/Corn": "Double Crops",
    "Dbl Crop Lettuce/Durum Wht": "Double Crops",
    "Dbl Crop Lettuce/Cantaloupe": "Double Crops",
    "Dbl Crop Lettuce/Cotton": "Double Crops",
    "Dbl Crop Lettuce/Barley": "Double Crops",
    "Dbl Crop Durum Wht/Sorghum": "Double Crops",
    "Dbl Crop Barley/Sorghum": "Double Crops",
    "Dbl Crop WinWht/Sorghum": "Double Crops",
    "Dbl Crop Barley/Corn": "Double Crops",
    "Dbl Crop WinWht/Cotton": "Double Crops",
    "Dbl Crop Soybeans/Cotton": "Double Crops",
    "Dbl Crop Soybeans/Oats": "Double Crops",
    "Dbl Crop Corn/Soybeans": "Double Crops",
    "Dbl Crop Barley/Soybeans": "Double Crops",

    "Fallow/Idle Cropland": "Fallowing",

    "Grassland/Pasture": "Pasture/Hay",
    "Pasture/Grass": "Pasture/Hay",
    "Grassland/Herbaceous": "Pasture/Hay",
    "Pasture/Hay": "Pasture/Hay",

    "Developed": "Urban Area",
    "Developed/Open Space": "Urban Area",
    "Developed/Low Intensity": "Urban Area",
    "Developed/Med Intensity": "Urban Area",
    "Developed/High Intensity": "Urban Area",

    "Christmas Trees": "Forest",
    "Forest": "Forest",
    "Deciduous Forest": "Forest",
    "Evergreen Forest": "Forest",
    "Mixed Forest": "Forest",

    "Shrubland": "Shrubland",
    "Barren": "Barren Land",

    "Water": "Waterbody",
    "Open Water": "Waterbody",
    "Woody Wetlands": "Waterbody",
    "Herbaceous Wetlands": "Waterbody",
    "Wetlands": "Waterbody",

    "Nonag/Undefined": "Other Land",
    "Aquaculture": "Other Land",
    "Perennial Ice/Snow": "Other Land",
    "No Crop": "Other Land",
    "Clouds/No Data": "Other Land"
}

In [ ]:
# Refinement for NLCD

def nlcd_refinement_class(nlcd_class):

    nlcd_class = str(nlcd_class).strip()

    urban = {
        "Developed, Open Space",
        "Developed, Low Intensity",
        "Developed, Medium Intensity",
        "Developed, High Intensity"
    }

    forest = {
        "Deciduous Forest",
        "Evergreen Forest",
        "Mixed Forest"
    }

    water = {
        "Open Water",
        "Woody Wetlands",
        "Emergent Herbaceous Wetlands"
    }
    
    if nlcd_class in urban:
        return "Urban Area"

    elif nlcd_class in forest:
        return "Forest"

    elif nlcd_class in water:
        return "Waterbody"

    else:
        return None


In [ ]:
cdl_matrices = {}
nlcd_matrices = {}

for year in years:

    cdl_file = f"{cdl_folder}/great_lakes_cdl_accuracy_{year}.xlsx"
    nlcd_file = f"{nlcd_folder}/nlcd_accuracy_{year}.xlsx"

    cdl = pd.read_excel(
        cdl_file,
        sheet_name="Average Error Matrix",
        index_col=0
    )

    cdl.index = cdl.index.astype(str).str.strip()
    cdl.columns = cdl.columns.astype(str).str.strip()
    
    # Make sure rows sum to 1
    cdl = cdl.div(cdl.sum(axis=1), axis=0)

    cdl_matrices[year] = cdl

    
    nlcd = pd.read_excel(
        nlcd_file,
        sheet_name="MC Matrix",
        index_col=0
    )

    nlcd.index = nlcd.index.astype(str).str.strip()
    nlcd.columns = nlcd.columns.astype(str).str.strip()

    nlcd = nlcd.div(nlcd.sum(axis=1), axis=0)

    nlcd_matrices[year] = nlcd

In [ ]:
samples = pd.read_csv(samples_file)

# Remove empty rows
samples = samples.dropna(subset=["CSBID"]).copy()

print("Number of valid samples:", len(samples))

In [ ]:
print("Rows:", len(samples))
print("Missing CSBID:", samples["CSBID"].isna().sum())
print("Text 'nan' CSBID:", samples["CSBID"].astype(str).str.lower().eq("nan").sum())

print(samples[["CSBID", "LULC_2008"]].head(10))

print(samples.columns.tolist())

In [ ]:
samples["Baseline_Typology"] = samples["Typology"]

In [ ]:
print(
    samples[
        ["CSBID", "Typology", "Baseline_Typology"]
    ].head(100)
)

In [ ]:
matches = (
    samples["Typology"]
    == samples["Baseline_Typology"]
)

print("Matched:", matches.sum())
print("Not matched:", (~matches).sum())

In [ ]:
rng = np.random.default_rng(42)

def simulate_class(mapped_class, matrix):
    mapped_class = str(mapped_class).strip()

    if mapped_class not in matrix.index:
        return mapped_class

    p = matrix.loc[mapped_class].astype(float)

    if p.sum() == 0:
        return mapped_class

    p = p / p.sum()

    return rng.choice(p.index, p=p.values)

In [ ]:
N_SIM = 1000
results = []

for _, row in samples.iterrows():

    original_typology = row["Baseline_Typology"]
    simulated_typologies = []

    for _ in range(N_SIM):

        sequence = []

        for year in years:

            original_cdl = row[f"CDL_{year}"]

            original_cdl_group = cdl_group.get(
                str(original_cdl).strip(),
                str(original_cdl).strip()
            )

            # If original CDL says Urban, Forest, or Waterbody,
            # evaluate uncertainty using NLCD
            if original_cdl_group in {"Urban Area", "Forest", "Waterbody"}:

                #Use NLCD uncertainity only
                sim_nlcd = simulate_class(
                    row[f"NLCD_{year}"],
                    nlcd_matrices[year]
                )

                nlcd_class = nlcd_refinement_class(sim_nlcd)

                # If simulated NLCD still supports Urban/Forest/Waterbody,
                # use the NLCD-derived class
                if nlcd_class is not None:
                    annual_class = nlcd_class

                # Otherwise go back to ORIGINAL CDL class
                else:
                    annual_class = original_cdl_group


            # For all other original CDL classes,
            # use CDL uncertainty only
            else:
                
                sim_cdl = simulate_class(
                    original_cdl,
                    cdl_matrices[year]
                )

                annual_class = cdl_group.get(
                    str(sim_cdl).strip(),
                        str(sim_cdl).strip()
                )

            sequence.append(annual_class)

        # Apply original transition typology rules
        simulated_typologies.append(
            assign_subtypology_label(sequence)
        )

    counts = Counter(simulated_typologies)

    result = {
        "CSBID": row["CSBID"],
        "Original_Typology": original_typology,
        "Stability": counts[original_typology] / N_SIM
    }

    # Add probabilities of all simulated typologies
    for typology, count in counts.items():
        result[typology] = count / N_SIM

    results.append(result)

In [ ]:
parcel_results = pd.DataFrame(results).fillna(0)
parcel_results

In [ ]:
class_results = (
    parcel_results
    .groupby("Original_Typology")
    .agg(
        Samples=("CSBID", "count"),
        Mean_Stability=("Stability", "mean")
    )
    .reset_index()
)

class_results

In [ ]:
out = "great_lakes_typology/monte_carlo_simulation"

class_results.to_csv(
    f"{out}/class_monte_carlo_stability.csv",
    index=False
)